# 02 — Seattle Commute Mode and Travel Time Trends

This notebook analyzes travel-to-work patterns for workers whose place of work is in
the Seattle-area POWPUMA (workplace super-PUMA), using ACS PUMS microdata downloaded
in notebook 01.

**Key outputs:**
1. Time trends in the **number** of Seattle-area workers by commute mode
2. Time trends in the **fraction** of Seattle-area workers by commute mode
3. Time trends in **mean travel time** by commute mode

All estimates use person weights (PWGTP) and standard errors use the 80 replicate
weights (PWGTP1–PWGTP80) with the successive difference replication method.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

DATA_DIR = os.path.abspath(os.path.join('..', 'data'))
RAW_DIR = os.path.join(DATA_DIR, 'raw', 'acs_pums')
DERIVED_DIR = os.path.join(DATA_DIR, 'derived')

YEARS_2010_PUMA = list(range(2013, 2020)) + [2021]
YEARS_2020_PUMA = [2022, 2023]
ALL_YEARS = YEARS_2010_PUMA + YEARS_2020_PUMA

REP_COLS = [f'PWGTP{i}' for i in range(1, 81)]

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Identify Seattle workplace PUMAs

POWPUMA (place-of-work PUMA) uses **super-PUMA** codes — coarser geographic units
than residential PUMAs. We identify the super-PUMA codes that cover Seattle
for both the 2010-based and 2020-based PUMA vintages.

In [ ]:
# Load 2010 PUMA names from gazetteer to show residential PUMAs
gaz_path = os.path.join(RAW_DIR, '2010_Gaz_PUMAs_national.txt')
gaz = pd.read_csv(gaz_path, sep='\t', dtype=str, encoding='latin-1')
gaz.columns = gaz.columns.str.strip()
wa_2010 = gaz[gaz['USPS'] == 'WA'].copy()
wa_2010['puma'] = wa_2010['GEOID'].str[2:].astype(int)

seattle_2010 = wa_2010[wa_2010['NAME'].str.contains('Seattle', case=False, na=False)]
print('2010-based Seattle residential PUMAs (11601–11605):')
for _, row in seattle_2010.iterrows():
    print(f"  {row['puma']:>5d}  {row['NAME']}")

print()
print('POWPUMA uses super-PUMA codes (coarser than residential PUMAs).')
print('Seattle super-PUMA for 2010-based geography: 11600')
print('Seattle super-PUMA for 2020-based geography: 23300')
print()

# 2020-based Seattle PUMAs (from Census 2020 PUMA Names PDF)
seattle_pumas_2020_names = {
    23312: 'Seattle City (West Seattle-Industrial)',
    23313: 'Seattle City (Southeast)',
    23314: 'Seattle City (Central)',
    23315: 'Seattle City (Lake Union-Downtown)',
    23316: 'Seattle City (Northwest)',
    23317: 'Seattle City (Northeast)',
    23318: 'Seattle City (North)',
}
print('2020-based Seattle residential PUMAs (23312–23318):')
for code, name in sorted(seattle_pumas_2020_names.items()):
    print(f'  {code:>5d}  {name}')

In [ ]:
# ------------------------------------------------------------------
# Define Seattle POWPUMA (workplace super-PUMA) sets
# ------------------------------------------------------------------
# POWPUMA codes are coarser than residential PUMAs. Each super-PUMA
# aggregates multiple residential PUMAs into one workplace area.
#
# 2010-based (2013–2021 ACS):
#   POWPUMA 11600 covers residential PUMAs 11601–11605 (all of Seattle)
#
# 2020-based (2022+ ACS):
#   POWPUMA 23300 covers residential PUMAs 23312–23318 (all of Seattle)

SEATTLE_POWPUMA_2010 = {11600}
SEATTLE_POWPUMA_2020 = {23300}

print(f'2010-based Seattle POWPUMA: {sorted(SEATTLE_POWPUMA_2010)}')
print(f'2020-based Seattle POWPUMA: {sorted(SEATTLE_POWPUMA_2020)}')


def get_seattle_powpumas(year):
    """Return the set of POWPUMA codes for Seattle given the ACS year."""
    return SEATTLE_POWPUMA_2010 if year <= 2021 else SEATTLE_POWPUMA_2020

## Harmonize commute mode variable

The ACS PUMS commute mode variable changed from **JWTR** (through 2018) to
**JWTRNS** (2019+), with different code values. We harmonize both into a
common set of mode categories. Vehicle occupancy (JWRIP / DRIVESP)
distinguishes drove-alone from carpool.

In [ ]:
# Pre-2019 JWTR codes → raw label
JWTR_MAP = {
    1: 'Car, truck, or van',
    2: 'Bus or trolley bus',
    3: 'Streetcar or trolley car',
    4: 'Subway or elevated',
    5: 'Railroad',
    6: 'Ferryboat',
    7: 'Taxicab',
    8: 'Motorcycle',
    9: 'Bicycle',
    10: 'Walked',
    11: 'Worked from home',
    12: 'Other method',
}

# Post-2019 JWTRNS codes → raw label
# Note: code 11 = Worked from home (verified by COVID-era spike to ~34%)
#        code 12 = Other method (consistently ~0.7%)
JWTRNS_MAP = {
    1: 'Car, truck, or van',
    2: 'Bus',
    3: 'Subway or elevated rail',
    4: 'Long-distance train or commuter rail',
    5: 'Light rail, streetcar, or trolley',
    6: 'Ferryboat',
    7: 'Taxicab',
    8: 'Motorcycle',
    9: 'Bicycle',
    10: 'Walked',
    11: 'Worked from home',
    12: 'Other method',
}

# Harmonized categories
def raw_to_harmonized(raw_label, is_carpool=False):
    """Map a raw Census mode label to a harmonized category."""
    if raw_label is None or pd.isna(raw_label):
        return None
    s = str(raw_label)
    if 'Car' in s or 'car' in s or 'Van' in s or 'van' in s or 'truck' in s:
        return 'Carpooled' if is_carpool else 'Drove alone'
    if 'Bus' in s or 'bus' in s:
        return 'Bus'
    if any(kw in s for kw in ('Subway', 'Railroad', 'rail', 'Light rail', 'Streetcar', 'trolley')):
        return 'Rail/streetcar'
    if 'Ferry' in s or 'ferry' in s:
        return 'Ferry'
    if 'Bicycle' in s or 'bicycle' in s:
        return 'Bicycle'
    if 'Walked' in s or 'walked' in s:
        return 'Walked'
    if 'home' in s.lower():
        return 'Worked from home'
    return 'Other'


def harmonize_modes(df):
    """Add 'mode' column with harmonized commute mode categories."""
    cols = set(df.columns)

    # Determine commute mode variable
    if 'JWTRNS' in cols:
        jwtr_col = 'JWTRNS'
        code_map = JWTRNS_MAP
    elif 'JWTR' in cols:
        jwtr_col = 'JWTR'
        code_map = JWTR_MAP
    else:
        raise ValueError(f'No commute mode variable found. Columns: {sorted(cols)}')

    # Determine carpool variable
    if 'DRIVESP' in cols:
        rip_col = 'DRIVESP'
    elif 'JWRIP' in cols:
        rip_col = 'JWRIP'
    else:
        rip_col = None

    # Map code → raw label
    df['mode_raw'] = df[jwtr_col].map(code_map)

    # Detect carpool vs drove alone
    is_car = df['mode_raw'].str.contains('Car|car|Van|van|truck', case=False, na=False)
    is_carpool = pd.Series(False, index=df.index)
    if rip_col:
        is_carpool = is_car & df[rip_col].notna() & (df[rip_col] > 1)

    df['mode'] = df['mode_raw'].apply(lambda x: raw_to_harmonized(x, is_carpool=False))
    df.loc[is_carpool, 'mode'] = 'Carpooled'

    return df


print('Mode harmonization functions defined.')

## Standard error computation with replicate weights

ACS PUMS uses successive difference replication with 80 replicate weights.

$$\text{SE}(\hat{\theta}) = \sqrt{\frac{4}{80} \sum_{r=1}^{80} (\hat{\theta}_r - \hat{\theta})^2}$$

In [ ]:
def replicate_se(est_full, est_reps):
    """Standard error via successive difference replication."""
    est_reps = np.asarray(est_reps, dtype=float)
    return np.sqrt((4.0 / 80.0) * np.nansum((est_reps - est_full) ** 2))


def weighted_total_with_se(weights, rep_weights_df):
    """Weighted total and SE."""
    total = weights.sum()
    reps = [rep_weights_df[c].sum() for c in REP_COLS if c in rep_weights_df.columns]
    se = replicate_se(total, reps)
    return total, se


def weighted_proportion_with_se(indicator, weights, rep_weights_df):
    """Weighted proportion and SE."""
    indicator = np.asarray(indicator, dtype=float)
    w = np.asarray(weights, dtype=float)
    p = np.sum(indicator * w) / np.sum(w)

    p_reps = []
    for c in REP_COLS:
        if c not in rep_weights_df.columns:
            continue
        wr = np.asarray(rep_weights_df[c], dtype=float)
        denom = np.sum(wr)
        p_reps.append(np.sum(indicator * wr) / denom if denom > 0 else np.nan)

    se = replicate_se(p, p_reps)
    return p, se


def weighted_mean_with_se(values, weights, rep_weights_df):
    """Weighted mean and SE."""
    mask = np.isfinite(values) & np.isfinite(weights)
    v = np.asarray(values[mask], dtype=float)
    w = np.asarray(weights[mask], dtype=float)
    mean_full = np.average(v, weights=w)

    rw = rep_weights_df.loc[mask]
    mean_reps = []
    for c in REP_COLS:
        if c not in rw.columns:
            continue
        wr = np.asarray(rw[c], dtype=float)
        if wr.sum() > 0:
            mean_reps.append(np.average(v, weights=wr))
        else:
            mean_reps.append(np.nan)

    se = replicate_se(mean_full, mean_reps)
    return mean_full, se


print('SE functions defined.')

## Load and filter data

Load all years, filter to workers whose place of work is in Seattle PUMAs,
and harmonize commute modes.

In [ ]:
frames = []
for year in ALL_YEARS:
    f = os.path.join(DERIVED_DIR, f'acs_pums_wa_commute_{year}.parquet')
    if not os.path.exists(f):
        print(f'{year}: file not found, skipping')
        continue

    df = pd.read_parquet(f)

    # Resolve PUMA column (may be PUMA, PUMA10, PUMA20, etc.)
    for candidate in ['PUMA', 'PUMA10', 'PUMA20', 'PUMA00']:
        if candidate in df.columns and candidate != 'PUMA':
            df['PUMA'] = df[candidate]
            break

    # Filter to workers in WA (POWSP == 53) whose workplace POWPUMA is in Seattle
    seattle_powpumas = get_seattle_powpumas(year)
    mask = (
        (df['POWSP'] == 53) &
        (df['POWPUMA'].isin(seattle_powpumas))
    )
    workers = df[mask].copy()

    # Harmonize modes
    workers = harmonize_modes(workers)
    workers['YEAR'] = year

    # Keep only needed columns
    keep = ['YEAR', 'PUMA', 'POWPUMA', 'mode', 'mode_raw', 'JWMNP', 'PWGTP'] + \
           [c for c in REP_COLS if c in workers.columns]
    workers = workers[[c for c in keep if c in workers.columns]]

    frames.append(workers)
    print(f'{year}: {len(workers):>6,} Seattle-area workers')

all_workers = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(all_workers):,} person-records across {all_workers["YEAR"].nunique()} years')

In [ ]:
# Quick check: mode distribution in most recent year
latest = all_workers[all_workers['YEAR'] == all_workers['YEAR'].max()]
mode_counts = latest.groupby('mode')['PWGTP'].sum().sort_values(ascending=False)
print(f'Mode distribution (weighted) — {latest["YEAR"].iloc[0]}:')
for mode, count in mode_counts.items():
    pct = 100 * count / mode_counts.sum()
    print(f'  {mode:<20s}  {count:>10,.0f}  ({pct:5.1f}%)')

## Compute mode share trends

For each year and mode, compute the weighted count (number of workers)
and fraction, along with standard errors from replicate weights.

In [ ]:
MODES_OF_INTEREST = [
    'Drove alone', 'Carpooled', 'Bus', 'Rail/streetcar', 'Ferry',
    'Bicycle', 'Walked', 'Worked from home', 'Other',
]

mode_trend_rows = []
for year in sorted(all_workers['YEAR'].unique()):
    yr_data = all_workers[all_workers['YEAR'] == year]
    total_weight = yr_data['PWGTP'].sum()
    rep_df = yr_data[REP_COLS] if REP_COLS[0] in yr_data.columns else pd.DataFrame()

    for mode in MODES_OF_INTEREST:
        indicator = (yr_data['mode'] == mode).astype(float)

        # Weighted count
        n_weighted = (indicator * yr_data['PWGTP']).sum()

        # Weighted fraction with SE
        if len(rep_df) > 0:
            frac, frac_se = weighted_proportion_with_se(
                indicator.values, yr_data['PWGTP'].values, rep_df
            )
            # SE for the count
            count_reps = []
            for c in REP_COLS:
                if c in rep_df.columns:
                    count_reps.append((indicator.values * rep_df[c].values).sum())
            count_se = replicate_se(n_weighted, count_reps)
        else:
            frac = n_weighted / total_weight if total_weight > 0 else 0
            frac_se = np.nan
            count_se = np.nan

        mode_trend_rows.append({
            'year': year,
            'mode': mode,
            'n_weighted': n_weighted,
            'n_weighted_se': count_se,
            'fraction': frac,
            'fraction_se': frac_se,
        })

mode_trends = pd.DataFrame(mode_trend_rows)
print(f'Mode trend table: {len(mode_trends)} rows')
mode_trends.head(12)

## Plot: Number of workers by commute mode over time

In [ ]:
# Focus on the major modes
major_modes = ['Drove alone', 'Bus', 'Worked from home', 'Walked',
               'Carpooled', 'Bicycle', 'Rail/streetcar']

fig, ax = plt.subplots(figsize=(12, 7))
for mode in major_modes:
    sub = mode_trends[mode_trends['mode'] == mode].sort_values('year')
    ax.errorbar(sub['year'], sub['n_weighted'], yerr=1.96 * sub['n_weighted_se'],
                marker='o', capsize=3, label=mode, linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Estimated number of workers')
ax.set_title('Number of Seattle-Area Workers by Commute Mode')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Mark PUMA geography break
ax.axvline(x=2021.5, color='gray', linestyle='--', alpha=0.5)
ax.text(2021.6, ax.get_ylim()[1] * 0.95, 'PUMA\nchange', fontsize=8, color='gray')

# Mark COVID gap
ax.axvspan(2019.5, 2020.5, alpha=0.1, color='red')
ax.text(2019.7, ax.get_ylim()[1] * 0.85, '2020\n(no data)', fontsize=8, color='red', alpha=0.7)

plt.tight_layout()
plt.show()

## Plot: Fraction of workers by commute mode over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
for mode in major_modes:
    sub = mode_trends[mode_trends['mode'] == mode].sort_values('year')
    ax.errorbar(sub['year'], 100 * sub['fraction'], yerr=100 * 1.96 * sub['fraction_se'],
                marker='o', capsize=3, label=mode, linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Percentage of workers (%)')
ax.set_title('Mode Share for Seattle-Area Workers Over Time')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

ax.axvline(x=2021.5, color='gray', linestyle='--', alpha=0.5)
ax.text(2021.6, ax.get_ylim()[1] * 0.95, 'PUMA\nchange', fontsize=8, color='gray')

ax.axvspan(2019.5, 2020.5, alpha=0.1, color='red')
ax.text(2019.7, ax.get_ylim()[1] * 0.85, '2020\n(no data)', fontsize=8, color='red', alpha=0.7)

plt.tight_layout()
plt.show()

## Compute travel time trends by mode

Mean travel time (JWMNP, in minutes) by commute mode and year, with SEs.

In [ ]:
# Exclude worked-from-home (travel time not applicable)
# JWMNP may be stored as object/string in some years; coerce to numeric
all_workers['JWMNP'] = pd.to_numeric(all_workers['JWMNP'], errors='coerce')

commuters = all_workers[
    (all_workers['mode'] != 'Worked from home') &
    (all_workers['JWMNP'].notna()) &
    (all_workers['JWMNP'] > 0)
].copy()

time_trend_rows = []
commute_modes = ['Drove alone', 'Bus', 'Carpooled', 'Bicycle', 'Walked', 'Rail/streetcar', 'Ferry']

for year in sorted(commuters['YEAR'].unique()):
    yr_data = commuters[commuters['YEAR'] == year]

    for mode in commute_modes:
        sub = yr_data[yr_data['mode'] == mode]
        if len(sub) < 10:
            continue

        rep_df = sub[REP_COLS] if REP_COLS[0] in sub.columns else pd.DataFrame()
        if len(rep_df) > 0:
            mean_time, time_se = weighted_mean_with_se(
                sub['JWMNP'].values, sub['PWGTP'].values, rep_df
            )
        else:
            mean_time = np.average(sub['JWMNP'], weights=sub['PWGTP'])
            time_se = np.nan

        time_trend_rows.append({
            'year': year,
            'mode': mode,
            'mean_travel_time_min': mean_time,
            'travel_time_se': time_se,
            'n_records': len(sub),
        })

time_trends = pd.DataFrame(time_trend_rows)
print(f'Travel time trend table: {len(time_trends)} rows')
time_trends.head(10)

## Plot: Mean travel time by commute mode over time

In [ ]:
plot_modes = ['Drove alone', 'Bus', 'Carpooled', 'Bicycle', 'Walked', 'Rail/streetcar']

fig, ax = plt.subplots(figsize=(12, 7))
for mode in plot_modes:
    sub = time_trends[time_trends['mode'] == mode].sort_values('year')
    if len(sub) == 0:
        continue
    ax.errorbar(sub['year'], sub['mean_travel_time_min'],
                yerr=1.96 * sub['travel_time_se'],
                marker='o', capsize=3, label=mode, linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Mean travel time (minutes)')
ax.set_title('Mean Commute Time to Seattle-Area Workplace by Mode')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

ax.axvline(x=2021.5, color='gray', linestyle='--', alpha=0.5)
ax.text(2021.6, ax.get_ylim()[1] * 0.95, 'PUMA\nchange', fontsize=8, color='gray')

ax.axvspan(2019.5, 2020.5, alpha=0.1, color='red')
ax.text(2019.7, ax.get_ylim()[1] * 0.85, '2020\n(no data)', fontsize=8, color='red', alpha=0.7)

plt.tight_layout()
plt.show()

## Summary table

Combined mode share and travel time summary for the most recent year.

In [ ]:
latest_year = mode_trends['year'].max()

summary = mode_trends[mode_trends['year'] == latest_year][['mode', 'n_weighted', 'fraction']].copy()
summary = summary.sort_values('n_weighted', ascending=False)

# Merge travel time for latest year
tt_latest = time_trends[time_trends['year'] == latest_year][['mode', 'mean_travel_time_min']]
summary = summary.merge(tt_latest, on='mode', how='left')

summary['fraction_pct'] = (100 * summary['fraction']).round(1)
summary['n_weighted'] = summary['n_weighted'].round(0).astype(int)
summary['mean_travel_time_min'] = summary['mean_travel_time_min'].round(1)

print(f'\nSummary for {latest_year}:')
print(summary[['mode', 'n_weighted', 'fraction_pct', 'mean_travel_time_min']]
      .rename(columns={
          'mode': 'Mode',
          'n_weighted': 'Workers (est.)',
          'fraction_pct': 'Share (%)',
          'mean_travel_time_min': 'Avg. time (min)',
      }).to_string(index=False))